# Ingest Litigations and Regional Judgements Data

This notebook ingests data from `Exploration-Lab/IL-TUR` and `Prarabdha/indian-legal-supervised-fine-tuning-data` into a Pinecone vector database.

In [ ]:
!pip install datasets pinecone sentence-transformers langchain-community langchain-huggingface python-dotenv huggingface_hub

In [1]:
import os
from dotenv import load_dotenv
from datasets import load_dataset
from pinecone import Pinecone, ServerlessSpec
from langchain_huggingface import HuggingFaceEmbeddings
import warnings

warnings.filterwarnings('ignore')

c:\Users\ksuni\Sunil\Personal\MTech-BITS\Sem-4\Dissertation\BlackBox-Regulatory-Gaurdian\Backend\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Authenticate with Hugging Face
Some datasets (like `Exploration-Lab/IL-TUR`) are gated. You must be authenticated to download them. Make sure to set `HF_TOKEN` in your `.env` file.

In [2]:
from huggingface_hub import login

load_dotenv("../.env")
hf_token = os.getenv("HF_TOKEN")
if hf_token:
    login(token=hf_token)
    print("Logged in to Hugging Face Hub successfully.")
else:
    print("Warning: HF_TOKEN not found in .env. You might need it for gated datasets.")


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Logged in to Hugging Face Hub successfully.


## 1. Setup Pinecone & Embeddings

In [3]:

PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
INDEX_NAME = "litigations-regional-judgements"
PINECONE_ENV = os.getenv("PINECONE_ENVIRONMENT", "us-east-1")

pc = Pinecone(api_key=PINECONE_API_KEY)

existing_indexes = [idx.name for idx in pc.list_indexes()]
if INDEX_NAME not in existing_indexes:
    print(f"Creating index {INDEX_NAME}...")
    pc.create_index(
        name=INDEX_NAME,
        dimension=384, # sentence-transformers/all-MiniLM-L6-v2 dimension
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region=PINECONE_ENV)
    )

index = pc.Index(INDEX_NAME)
print("Pinecone Index ready.")

Pinecone Index ready.


In [4]:
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5622.68it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


## 2. Ingest `Exploration-Lab/IL-TUR`
The Indian Legal Text Understanding Benchmark.

In [7]:
from datasets import concatenate_datasets
print("Loading Exploration-Lab/IL-TUR dataset...")
dataset = load_dataset("Exploration-Lab/IL-TUR", "lner")
il_tur = concatenate_datasets([dataset["fold_1"], dataset["fold_2"], dataset["fold_3"]])
print(f"Loaded {len(il_tur)} samples.")

Loading Exploration-Lab/IL-TUR dataset...
Loaded 105 samples.


In [8]:
def process_il_tur(dataset, batch_size=100):
    vectors_to_upsert = []
    for i, item in enumerate(dataset):
        text = item.get("text", "")
        if not text:
            text = item.get("document", "")
        if not text:
            text = " ".join([str(v) for k,v in item.items() if isinstance(v, str)])
        
        embedding = embeddings.embed_query(text)
        
        metadata = {
            "source": "il_tur",
            "text": text[:5000] # Truncate text for metadata to avoid Pinecone size limits
        }
        
        for key in ["label", "category", "id"]:
            if key in item:
                metadata[key] = str(item[key])
        
        vectors_to_upsert.append((f"iltur_{i}", embedding, metadata))
        
        if len(vectors_to_upsert) >= batch_size:
            index.upsert(vectors=vectors_to_upsert)
            vectors_to_upsert = []
            print(f"Upserted {i+1} records from IL-TUR...")
            
    if vectors_to_upsert:
        index.upsert(vectors=vectors_to_upsert)
    print("Finished ingesting IL-TUR.")

# process_il_tur(il_tur) # Uncomment to run

## 3. Ingest `Prarabdha/indian-legal-supervised-fine-tuning-data`
Indian legal supervised fine-tuning data.

In [9]:
print("Loading Prarabdha/indian-legal-supervised-fine-tuning-data dataset...")
sft_data = load_dataset("Prarabdha/indian-legal-supervised-fine-tuning-data", split="train")
print(f"Loaded {len(sft_data)} samples.")

Loading Prarabdha/indian-legal-supervised-fine-tuning-data dataset...


Generating train split: 100%|██████████| 6055371/6055371 [00:21<00:00, 285100.99 examples/s]


Loaded 6055371 samples.


In [10]:
def process_sft_data(dataset, batch_size=100):
    vectors_to_upsert = []
    for i, item in enumerate(dataset):
        text = ""
        if "instruction" in item and "output" in item:
            text = f"Instruction: {item['instruction']} \nOutput: {item['output']}"
        elif "prompt" in item and "completion" in item:
            text = f"Prompt: {item['prompt']} \nCompletion: {item['completion']}"
        else:
            text = " ".join([str(v) for k,v in item.items() if isinstance(v, str)])
        
        embedding = embeddings.embed_query(text)
        
        metadata = {
            "source": "prarabdha_sft",
            "text": text[:5000]
        }
        
        vectors_to_upsert.append((f"sft_{i}", embedding, metadata))
        
        if len(vectors_to_upsert) >= batch_size:
            index.upsert(vectors=vectors_to_upsert)
            vectors_to_upsert = []
            print(f"Upserted {i+1} records from SFT data...")
            
    if vectors_to_upsert:
        index.upsert(vectors=vectors_to_upsert)
    print("Finished ingesting Prarabdha SFT data.")

process_sft_data(sft_data) # Uncomment to run

Upserted 100 records from SFT data...
Upserted 200 records from SFT data...
Upserted 300 records from SFT data...
Upserted 400 records from SFT data...
Upserted 500 records from SFT data...
Upserted 600 records from SFT data...
Upserted 700 records from SFT data...
Upserted 800 records from SFT data...
Upserted 900 records from SFT data...
Upserted 1000 records from SFT data...
Upserted 1100 records from SFT data...
Upserted 1200 records from SFT data...
Upserted 1300 records from SFT data...
Upserted 1400 records from SFT data...
Upserted 1500 records from SFT data...
Upserted 1600 records from SFT data...
Upserted 1700 records from SFT data...
Upserted 1800 records from SFT data...
Upserted 1900 records from SFT data...
Upserted 2000 records from SFT data...
Upserted 2100 records from SFT data...
Upserted 2200 records from SFT data...
Upserted 2300 records from SFT data...
Upserted 2400 records from SFT data...
Upserted 2500 records from SFT data...
Upserted 2600 records from SFT dat

MaxRetryError: HTTPSConnectionPool(host='litigations-regional-judgements-0u9f7j1.svc.aped-4627-b74a.pinecone.io', port=443): Max retries exceeded with url: /vectors/upsert (Caused by NameResolutionError("HTTPSConnection(host='litigations-regional-judgements-0u9f7j1.svc.aped-4627-b74a.pinecone.io', port=443): Failed to resolve 'litigations-regional-judgements-0u9f7j1.svc.aped-4627-b74a.pinecone.io' ([Errno 11001] getaddrinfo failed)"))